# Esempio di Training e Predizione con Rete Neurale

Questo notebook mostra come usare il modulo predictor per:
1. Caricare e preprocessare dati
2. Creare e trainare una rete neurale
3. Valutare le performance
4. Fare predizioni

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from pathlib import Path

# Aggiungi src al path
sys.path.insert(0, '../src')

from data import DataPreprocessor, MachineryDataset
from models import create_medium_model
from training import ModelTrainer
from utils import plot_training_history, plot_predictions, calculate_metrics, print_metrics

print("Librerie importate con successo!")

## 1. Genera Dati Sintetici (per test)

In produzione, sostituisci questa sezione caricando i tuoi dati reali.

In [ ]:
# Genera dati sintetici per test
np.random.seed(42)

n_samples = 1000
n_inputs = 5
n_outputs = 3

# Input casuali
X = np.random.randn(n_samples, n_inputs) * 10 + 50

# Output = funzione lineare di X + rumore
# In produzione, questi saranno i tuoi dati reali del macchinario
weights = np.random.randn(n_inputs, n_outputs)
y = X @ weights + np.random.randn(n_samples, n_outputs) * 2

print(f"Dati generati: {X.shape[0]} campioni")
print(f"Input features: {X.shape[1]}")
print(f"Output features: {y.shape[1]}")

## 2. Preprocessing

In [ ]:
# Crea preprocessor
preprocessor = DataPreprocessor(scaling_method='standard')

# Split train/val/test
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=42
)

# Scala i dati
X_train_scaled, y_train_scaled = preprocessor.fit_transform(X_train, y_train)
X_val_scaled, y_val_scaled = preprocessor.transform(X_val, y_val)
X_test_scaled, y_test_scaled = preprocessor.transform(X_test, y_test)

print(f"Train: {len(X_train)} campioni")
print(f"Validation: {len(X_val)} campioni")
print(f"Test: {len(X_test)} campioni")

## 3. Crea Dataset e DataLoader

In [ ]:
# Dataset PyTorch
train_dataset = MachineryDataset(X_train_scaled, y_train_scaled)
val_dataset = MachineryDataset(X_val_scaled, y_val_scaled)
test_dataset = MachineryDataset(X_test_scaled, y_test_scaled)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Dataset creati!")
print(f"Input dim: {train_dataset.get_input_dim()}")
print(f"Output dim: {train_dataset.get_output_dim()}")

## 4. Crea e Traína il Modello

In [ ]:
# Crea modello
model = create_medium_model(
    input_size=train_dataset.get_input_dim(),
    output_size=train_dataset.get_output_dim()
)

print(f"Modello creato!")
print(f"Parametri totali: {sum(p.numel() for p in model.parameters())}")
print(model)

In [ ]:
# Setup trainer
trainer = ModelTrainer(
    model,
    learning_rate=0.001,
    loss_fn='mse'
)

# Training
history = trainer.train(
    train_loader,
    val_loader,
    epochs=100,
    patience=15,
    save_dir='../checkpoints'
)

## 5. Visualizza Training History

In [ ]:
plot_training_history(
    history['train_losses'],
    history['val_losses']
)

## 6. Valuta sul Test Set

In [ ]:
# Predizioni
y_pred_scaled = trainer.predict(X_test_scaled)
y_pred = preprocessor.inverse_transform_output(y_pred_scaled)

# Calcola metriche
metrics = calculate_metrics(
    y_test,
    y_pred,
    output_names=['Output 1', 'Output 2', 'Output 3']
)

print_metrics(metrics)

## 7. Visualizza Predizioni

In [ ]:
plot_predictions(
    y_test,
    y_pred,
    output_names=['Pressione', 'Temperatura', 'Velocità']
)

## 8. Fai una Predizione su Nuovi Dati

In [ ]:
# Nuovi parametri operativi
new_input = np.array([
    [45.2, 50.1, 55.3, 48.7, 52.1],  # Esempio 1
    [60.5, 45.8, 50.2, 55.9, 48.3],  # Esempio 2
])

# Preprocessa
new_input_scaled = preprocessor.transform(new_input)

# Predici
new_pred_scaled = trainer.predict(new_input_scaled)
new_pred = preprocessor.inverse_transform_output(new_pred_scaled)

# Mostra risultati
print("Nuove predizioni:")
for i, pred in enumerate(new_pred):
    print(f"\nEsempio {i+1}:")
    print(f"  Input: {new_input[i]}")
    print(f"  Output predetto: {pred}")

## 9. Salva il Modello per Uso Futuro

In [ ]:
# Il modello è già stato salvato durante il training
# Salviamo anche gli scaler
preprocessor.save_scalers('../checkpoints/scalers.pkl')

print("Modello e scaler salvati in ../checkpoints/")